In [1]:
# ============================================================
# SQL QUERY GENERATOR - COMPLETE CODE FOR GOOGLE COLAB
# ============================================================

# Install required packages
!pip install -q gradio
!pip install -q transformers
!pip install -q torch

import gradio as gr
from datetime import datetime
import re

# ============================================================
# CLASS 1: NLP PROCESSOR - Understand Natural Language
# ============================================================

class NLPProcessor:
    def __init__(self):
        # Map keywords to database table names
        self.table_keywords = {
            'user': 'users',
            'users': 'users',
            'customer': 'users',
            'account': 'users',
            'member': 'users',
            'transaction': 'transactions',
            'transactions': 'transactions',
            'payment': 'transactions',
            'purchase': 'transactions',
            'order': 'orders',
            'orders': 'orders',
            'product': 'products',
            'products': 'products',
            'item': 'products',
            'employee': 'employees',
            'employees': 'employees',
            'staff': 'employees',
            'worker': 'employees',
        }

        # Map keywords to column names
        self.column_keywords = {
            'name': 'name',
            'email': 'email',
            'phone': 'phone',
            'signup': 'signup_date',
            'signed': 'signup_date',
            'created': 'created_at',
            'amount': 'amount',
            'price': 'price',
            'quantity': 'quantity',
            'total': 'total',
            'status': 'status',
            'date': 'date',
            'department': 'department',
            'salary': 'salary',
            'description': 'description',
            'address': 'address',
        }

        # Map date keywords to days
        self.date_keywords = {
            'today': 0,
            'yesterday': 1,
            'week': 7,
            'last week': 7,
            'month': 30,
            'last month': 30,
            'year': 365,
            'last year': 365,
        }

    def process(self, user_input):
        """Main function to process natural language input"""
        try:
            input_lower = user_input.lower()

            # Step 1: Detect which table to query
            tables = self._detect_tables(input_lower)
            if not tables:
                return {
                    "success": False,
                    "error": "Could not detect database table. Try mentioning: users, transactions, products, orders, or employees"
                }

            # Step 2: Detect which columns to select
            columns = self._detect_columns(input_lower)

            # Step 3: Detect WHERE conditions (amounts, status, etc.)
            conditions = self._detect_conditions(input_lower)

            # Step 4: Detect date ranges
            date_filters = self._detect_date_filters(input_lower)

            # Step 5: Create summary text
            summary = self._create_summary(tables, columns, conditions, date_filters)

            return {
                "success": True,
                "tables": tables,
                "columns": columns,
                "conditions": conditions,
                "date_filters": date_filters,
                "summary": summary
            }
        except Exception as e:
            return {"success": False, "error": str(e)}

    def _detect_tables(self, text):
        """Find which tables are mentioned"""
        tables = []
        for keyword, table in self.table_keywords.items():
            if keyword in text and table not in tables:
                tables.append(table)
        return tables

    def _detect_columns(self, text):
        """Find which columns are mentioned"""
        columns = []
        for keyword, column in self.column_keywords.items():
            if keyword in text and column not in columns:
                columns.append(column)
        return columns if columns else ['*']

    def _detect_conditions(self, text):
        """Find WHERE conditions like amounts, status, etc."""
        conditions = {}

        # Find amount conditions (greater than, exceed, over, etc.)
        amount_pattern = r'(exceed|over|greater than|more than|above|>)\s+(\$)?([\d,]+)'
        amount_match = re.search(amount_pattern, text, re.IGNORECASE)
        if amount_match:
            amount = amount_match.group(3).replace(',', '')
            conditions['amount'] = float(amount)
            conditions['amount_operator'] = '>'

        # Find status conditions
        statuses = ['active', 'inactive', 'pending', 'completed', 'failed', 'approved']
        for status in statuses:
            if status in text:
                conditions['status'] = status
                break

        return conditions

    def _detect_date_filters(self, text):
        """Find date ranges"""
        date_filters = {}
        for keyword, days in self.date_keywords.items():
            if keyword in text:
                date_filters['range'] = keyword
                date_filters['days'] = days
                break
        return date_filters

    def _create_summary(self, tables, columns, conditions, date_filters):
        """Create a summary of what was detected"""
        parts = []

        if tables:
            parts.append(f"📊 Tables: {', '.join(tables)}")

        if columns and columns != ['*']:
            parts.append(f"📋 Columns: {', '.join(columns)}")

        if conditions:
            cond_parts = []
            if 'amount' in conditions:
                cond_parts.append(f"Amount > ${conditions['amount']}")
            if 'status' in conditions:
                cond_parts.append(f"Status = {conditions['status']}")
            if cond_parts:
                parts.append(f"🔍 Conditions: {', '.join(cond_parts)}")

        if date_filters:
            parts.append(f"📅 Date: Within {date_filters['range']}")

        return "\n".join(parts) if parts else "No specific conditions detected"

# ============================================================
# CLASS 2: SQL GENERATOR - Build SQL Query
# ============================================================

class SQLGenerator:
    def __init__(self):
        self.default_limit = 1000

    def generate(self, tables, columns, conditions, date_filters):
        """Generate SQL query from components"""
        try:
            if not tables:
                return "-- Error: No tables specified\n-- Please mention a table like: users, transactions, products, orders"

            # Select the first table
            table = tables[0]

            # Build SELECT clause
            if columns and columns != ['*']:
                cols = ", ".join(columns)
            else:
                cols = "*"

            query = f"SELECT {cols}\nFROM {table}"

            # Build WHERE clause
            where_clauses = []

            # Add amount condition
            if conditions.get('amount'):
                operator = conditions.get('amount_operator', '>')
                amount = conditions['amount']
                where_clauses.append(f"amount {operator} {amount}")

            # Add status condition
            if conditions.get('status'):
                status = conditions['status']
                where_clauses.append(f"status = '{status}'")

            # Add date condition
            if date_filters.get('days') is not None:
                days = date_filters['days']
                if days == 0:
                    where_clauses.append("DATE(date) = DATE('now')")
                else:
                    where_clauses.append(f"date >= DATE('now', '-{days} days')")

            # Combine WHERE clauses
            if where_clauses:
                query += "\nWHERE " + "\nAND ".join(where_clauses)

            # Add LIMIT for safety
            query += f"\nLIMIT {self.default_limit};"

            return query
        except Exception as e:
            return f"-- Error: {str(e)}"

# ============================================================
# CLASS 3: SECURITY VALIDATOR - Block Dangerous Queries
# ============================================================

class SecurityValidator:
    def __init__(self):
        self.dangerous_keywords = [
            'DROP', 'DELETE', 'ALTER', 'TRUNCATE', 'INSERT', 'UPDATE',
            'EXEC', 'EXECUTE', 'SCRIPT', 'UNION', 'DECLARE', 'SET',
            'CAST', 'CONVERT', 'REPLACE', 'CREATE', 'GRANT', 'REVOKE'
        ]

    def is_safe(self, user_input):
        """Check if input contains dangerous SQL keywords"""
        input_upper = user_input.upper()

        # Check for dangerous keywords
        for keyword in self.dangerous_keywords:
            if keyword in input_upper:
                return False, f"Dangerous keyword detected: {keyword}"

        # Check for SQL injection patterns
        injection_patterns = [r"';", r"--", r"/\*", r"\*/"]
        for pattern in injection_patterns:
            if re.search(pattern, user_input, re.IGNORECASE):
                return False, "SQL injection pattern detected"

        return True, "Safe"

# ============================================================
# CLASS 4: QUERY HISTORY - Store Previous Queries
# ============================================================

class QueryHistory:
    def __init__(self):
        self.history = []

    def add(self, user_input, sql_query):
        """Add query to history"""
        self.history.append({
            "input": user_input,
            "query": sql_query,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    def get_all(self):
        """Get all queries"""
        return self.history

    def clear(self):
        """Clear history"""
        self.history = []

    def get_formatted(self):
        """Get formatted history for display"""
        if not self.history:
            return "📋 No queries yet. Generate your first query!"

        text = "📋 QUERY HISTORY:\n\n"
        for i, record in enumerate(self.history, 1):
            text += f"{i}. INPUT: {record['input']}\n"
            text += f"   QUERY: {record['query']}\n"
            text += f"   TIME: {record['timestamp']}\n"
            text += "-" * 80 + "\n\n"
        return text

# ============================================================
# INITIALIZE ALL COMPONENTS
# ============================================================

nlp_processor = NLPProcessor()
sql_generator = SQLGenerator()
security_validator = SecurityValidator()
query_history = QueryHistory()

# ============================================================
# MAIN FUNCTIONS FOR GRADIO INTERFACE
# ============================================================

def generate_sql_query(user_input):
    """Main function to generate SQL from natural language"""
    try:
        # Check for security threats
        is_safe, message = security_validator.is_safe(user_input)
        if not is_safe:
            return "🚨 SECURITY WARNING!", message, "", ""

        # Process natural language
        processed = nlp_processor.process(user_input)

        if not processed.get("success"):
            return "❌ ERROR", processed.get("error", "Unknown error"), "", ""

        # Generate SQL
        sql_query = sql_generator.generate(
            tables=processed.get("tables", []),
            columns=processed.get("columns", []),
            conditions=processed.get("conditions", {}),
            date_filters=processed.get("date_filters", {})
        )

        # Add to history
        query_history.add(user_input, sql_query)

        # Return results
        status = "✅ SUCCESS! Query generated!"
        summary = processed.get("summary", "")

        return status, summary, sql_query, ""

    except Exception as e:
        return "❌ ERROR", str(e), "", ""

def get_history_display():
    """Get history for display"""
    return query_history.get_formatted()

def clear_history_func():
    """Clear history"""
    query_history.clear()
    return "✅ History cleared!"

# ============================================================
# CREATE GRADIO INTERFACE
# ============================================================

with gr.Blocks(title="SQL Query Generator", theme=gr.themes.Soft()) as demo:

    # Header
    gr.Markdown("""
    # 🗄️ SQL Query Generator
    ### Convert Natural Language to SQL
    Type what you want in plain English, and get SQL code instantly!
    """)

    # Main Tabs
    with gr.Tabs():

        # ========== TAB 1: QUERY GENERATOR ==========
        with gr.Tab("🚀 Generate Query"):
            with gr.Row():
                # Left Column: Input
                with gr.Column(scale=1):
                    gr.Markdown("### 📝 Your Request")
                    user_input = gr.Textbox(
                        label="What SQL do you need?",
                        placeholder="Examples:\n• Find users from last month\n• Get transactions over $1000\n• Show products by price",
                        lines=5,
                        max_lines=10
                    )
                    generate_btn = gr.Button("🚀 Generate SQL", variant="primary", size="lg")

                # Right Column: Output
                with gr.Column(scale=1):
                    gr.Markdown("### 📊 Results")
                    status_output = gr.Textbox(
                        label="Status",
                        interactive=False,
                        lines=1
                    )
                    summary_output = gr.Textbox(
                        label="What was detected",
                        interactive=False,
                        lines=4
                    )
                    sql_output = gr.Code(
                        label="Your SQL Query",
                        language="sql",
                        interactive=False,
                        lines=8
                    )
                    error_output = gr.Textbox(
                        label="Errors (if any)",
                        interactive=False,
                        lines=2
                    )

            # Connect button to function
            generate_btn.click(
                fn=generate_sql_query,
                inputs=user_input,
                outputs=[status_output, summary_output, sql_output, error_output]
            )

        # ========== TAB 2: EXAMPLES ==========
        with gr.Tab("📚 Examples"):
            gr.Markdown("""
            ## 💡 Try These Example Queries:

            ### Users Table
            - "Find all users who signed up last month"
            - "Get active users"
            - "Show users with email"

            ### Transactions Table
            - "Get transactions that exceed $1000"
            - "Show pending transactions from last week"
            - "Find payments from this month"

            ### Products Table
            - "Show products with price greater than $100"
            - "Get active products"
            - "List items created last year"

            ### Orders Table
            - "Find orders from last week"
            - "Show completed orders"
            - "Get all order details"

            ### Employees Table
            - "List employees in sales department"
            - "Show employees with salary above $50000"
            - "Get staff from this month"

            ## 🔒 Safety Features
            ✅ Automatic SQL Injection Prevention
            ✅ Dangerous Keyword Detection
            ✅ Automatic LIMIT clauses
            ✅ Input Validation
            """)

        # ========== TAB 3: HISTORY ==========
        with gr.Tab("📋 History"):
            history_output = gr.Textbox(
                label="All Previous Queries",
                lines=12,
                interactive=False
            )

            with gr.Row():
                refresh_btn = gr.Button("🔄 Refresh", variant="secondary")
                clear_btn = gr.Button("🗑️ Clear History", variant="stop")

            # Show initial history
            history_output.value = query_history.get_formatted()

            # Connect buttons
            refresh_btn.click(fn=get_history_display, outputs=history_output)
            clear_btn.click(fn=clear_history_func, outputs=history_output)

# ============================================================
# LAUNCH THE APPLICATION
# ============================================================

print("🚀 Starting SQL Query Generator...")
print("=" * 60)

demo.launch(share=True)

print("=" * 60)
print("✅ SQL Query Generator is running!")
print("Click the link above to open the interface")

/tmp/ipykernel_1074/4024149195.py:365: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SQL Query Generator", theme=gr.themes.Soft()) as demo:


🚀 Starting SQL Query Generator...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6fdcf675af060e37ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ SQL Query Generator is running!
Click the link above to open the interface
